# 第 1 周末练习 —— 技术问答解释器

## 练习目标（理念）

为展示你对 **OpenAI API** 与 **Ollama** 的熟悉程度，请构建一个工具：

- **输入**：多行粘贴一段待解释的代码（空行结束）
- **输出**：清晰、分步的解释（system 定「怎么讲」，user 包代码）
- **双后端对比**：同一套 prompts，分别走云端 `gpt-4o-mini` 与本地 `llama3.2`，且都用 **流式** 边生成边打印

这是你在课程期间可以亲自使用的工具！

## 和本课 Day 1 / Day 2 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Chat Completions API | `openai.chat.completions.create(..., stream=True)` |
| `messages`（system / user） | `SYSTEM_PROMPT` + `create_user_prompt(...)` |
| 流式输出 `stream=True` | 逐块 `print(..., end="", flush=True)` |
| Ollama Python 库 | `ollama.chat(..., stream=True)` |
| 环境变量 | `OPENAI_API_KEY`（缺失时用 `getpass` 交互输入） |

## 怎么跑

1. 准备 `.env` 的 `OPENAI_API_KEY`；本机安装并启动 Ollama，已 `pull llama3.2`
2. 从上到下运行；输入代码格会等待你多行粘贴，**空行结束**
3. 再分别跑 GPT 流式格与 Llama 流式格，对比风格


In [ ]:
# ========== 导入：OpenAI SDK + dotenv + ollama 库 ==========

# 第 1 周练习：逻辑与字符串保持原样
# 导入标准库 os：读/写环境变量（Environment Variables）
import os
# 从 openai 导入 OpenAI：调用云端 Chat Completions
from openai import OpenAI
# 从 dotenv 导入 load_dotenv：把 .env 读进进程环境
from dotenv import load_dotenv
# 导入 ollama 官方 Python 包：直接调本地模型（不是 OpenAI 兼容 HTTP 客户端）
import ollama


In [ ]:
# ========== 常量：两个后端的模型名 ==========

# 第 1 周练习：逻辑与字符串保持原样
# 常量

# 云端 OpenAI 小模型：便宜、适合解释类问答
MODEL_GPT = 'gpt-4o-mini'
# 本地 Ollama 模型名：须与本机已安装一致
MODEL_LLAMA = 'llama3.2'


In [ ]:
# ========== 环境配置：加载密钥并创建 OpenAI 客户端 ==========

# 第 1 周练习：逻辑与字符串保持原样
# 配置环境
# 加载 .env（默认不 override；进程里已有的同名变量优先）
load_dotenv()

# 尝试从环境读取 OPENAI_API_KEY
api_key = os.getenv('OPENAI_API_KEY')
# 若缺失：交互式让用户输入，并写回当前进程环境
if not api_key:
    # getpass：输入时不回显，降低密钥被肩窥风险
    import getpass
    # 提示文案保持英文原样
    print('OPENAI_API_KEY not found in environment. Please enter it:')
    api_key = getpass.getpass()
    # 写入 os.environ，供后续 OpenAI() 默认读取
    os.environ['OPENAI_API_KEY'] = api_key

# 使用默认配置创建客户端（会从环境读 OPENAI_API_KEY）
openai = OpenAI()


In [ ]:
# ========== 提示词：system 人设 + 把代码包进 user 的小函数 ==========

# 第 1 周练习：逻辑与字符串保持原样
# SYSTEM_PROMPT 保持英文：这是发给模型的指令，改译会改变回答风格
SYSTEM_PROMPT = """
You are a helpful Python expert capable of explaining complex code snippets.
Please provide clear, concise explanations and break down how the code works step-by-step.
Focus on both the 'what' and the 'why' to help developers understand the reasoning behind the implementation.
if yo
"""

def create_user_prompt(code_snippet):
    # 把代码片段嵌进固定英文模板，形成可复用的 user 消息
    """Wraps the code snippet into a reusable prompt structure."""
    return f"Please explain what this code does and why:\n\n{code_snippet}"


In [ ]:
# ========== 交互输入：多行读入代码，空行结束 ==========

# 第 1 周练习：逻辑与字符串保持原样
# Ask the user for the code snippet they want explained
# lines：逐行收集用户输入的代码
lines = []
while True:
    # input()：从标准输入读一行（在 Jupyter 里会弹出输入框）
    line = input()
    # 空行视为结束（注意：代码中间的空行也会打断；逻辑保持原样）
    if not line:
        break
    lines.append(line)

# 用换行拼回完整代码片段
code_snippet = "\n".join(lines)
# 套进 user prompt 模板，供后面两个后端共用
user_prompt = create_user_prompt(code_snippet)



In [ ]:
# ========== 云端 GPT：流式 Chat Completions 边收边打印 ==========

# 第 1 周练习：逻辑与字符串保持原样
# 用 gpt-4o-mini 流式回答
# stream=True：返回可迭代的增量事件，而不是等整段生成完
responses = openai.chat.completions.create(
    model=MODEL_GPT,
    messages=[
        # system：解释风格与侧重点
        {"role": "system", "content": SYSTEM_PROMPT},
        # user：具体代码问题
        {"role": "user", "content": user_prompt}
    ],
    stream=True
)

# 遍历每个流式 chunk；增量文本在 choices[0].delta.content
for chunk in responses:
    if chunk.choices[0].delta.content:
        # end=""：不换行拼接；flush=True：立刻刷到屏幕，体现「流式」
        print(chunk.choices[0].delta.content, end="", flush=True)
# 全部结束后补一个换行，避免提示符粘在同一行
print()


In [ ]:
# ========== 本地 Llama：ollama.chat 流式打印 ==========

# 第 1 周练习：逻辑与字符串保持原样
# 用 Llama 3.2 回答
# ollama.chat：走 ollama Python 库；同样传 system/user 与 stream=True
stream = ollama.chat(
    model=MODEL_LLAMA,
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt}
    ],
    stream=True
)

# 每个 chunk 是字典；增量在 message.content
for chunk in stream:
    print(chunk["message"]["content"], end="", flush=True)
# 结束后换行收尾
print()
